le nom: El jattioui 
le prenom: Maryame 
Master:GLCC

In [1]:
import numpy as np

# X = [Heures d'études, Projets réalisés]
X = np.array([[10, 1], [20, 2], [5, 0], [15, 2], [2, 0]])
# y = Note à l'examen (Cible)
y = np.array([12, 18, 7, 15, 5])

# =================================================================
# 1. STRUCTURE DE L'ARBRE DE RÉGRESSION (Base Learner)
# =================================================================
class XGBoostNode:
    """ 
    Représente un nœud dans un arbre de boosting.
    Contrairement à un arbre classique, chaque feuille stocke une 'valeur de poids'
    qui sert de correction aux prédictions précédentes.
    """
    def __init__(self, feature=None, threshold=None, left=None, right=None, value=None):
        self.feature = feature     # Index de la caractéristique testée
        self.threshold = threshold # Valeur de coupure (split)
        self.left = left           # Branche gauche
        self.right = right         # Branche droite
        self.value = value         # Poids de la feuille (utilisé uniquement si c'est une feuille)

class XGBoostTree:
    """ 
    Arbre de décision individuel (Weak Learner) entraîné pour prédire 
    les résidus (erreurs) du modèle global.
    """
    def __init__(self, max_depth=3):
        self.max_depth = max_depth
        self.root = None

    def fit(self, X, residuals):
        """ Apprend à prédire les erreurs (residuals) """
        self.root = self._grow_tree(X, residuals)

    def _grow_tree(self, X, residuals, depth=0):
        n_samples = len(X)
        
        # CONDITION D'ARRÊT : Profondeur max ou trop peu de données
        # Si on s'arrête, on calcule la valeur optimale de la feuille.
        # En XGBoost simplifié (Régression), c'est la moyenne des résidus.
        if depth >= self.max_depth or n_samples < 2:
            return XGBoostNode(value=np.mean(residuals))

        # RECHERCHE DU MEILLEUR SPLIT (DIVISION)
        # On cherche la variable et le seuil qui réduisent le plus la variance
        best_feat, best_thresh = self._find_best_split(X, residuals)
        
        # Séparation effective des données en deux groupes
        left_idx = X[:, best_feat] <= best_thresh
        right_idx = X[:, best_feat] > best_thresh

        # Si le split ne sépare rien, on crée une feuille
        if sum(left_idx) == 0 or sum(right_idx) == 0:
            return XGBoostNode(value=np.mean(residuals))

        # APPELS RÉCURSIFS : On construit les sous-arbres
        left = self._grow_tree(X[left_idx], residuals[left_idx], depth + 1)
        right = self._grow_tree(X[right_idx], residuals[right_idx], depth + 1)
        
        return XGBoostNode(feature=best_feat, threshold=best_thresh, left=left, right=right)

    def _find_best_split(self, X, residuals):
        """ Parcourt les colonnes pour trouver la question qui réduit le plus l'erreur """
        best_variance_reduction = -1
        split_idx, split_thresh = 0, 0
        
        for f_idx in range(X.shape[1]):
            thresholds = np.unique(X[:, f_idx])
            for t in thresholds:
                # On simule la séparation
                mask = X[:, f_idx] <= t
                # On calcule la réduction de variance (Gain)
                # Gain = Var(Parent) - [Poids_Gauche * Var(Gauche) + Poids_Droite * Var(Droite)]
                reduction = self._variance_reduction(residuals, mask)
                
                if reduction > best_variance_reduction:
                    best_variance_reduction = reduction
                    split_idx = f_idx
                    split_thresh = t
        return split_idx, split_thresh

    def _variance_reduction(self, residuals, mask):
        """ Calcule le gain en pureté après une division """
        parent_var = np.var(residuals)
        n = len(residuals)
        n_l, n_r = sum(mask), sum(~mask)
        
        if n_l == 0 or n_r == 0: return 0
        
        child_var = (n_l/n) * np.var(residuals[mask]) + (n_r/n) * np.var(residuals[~mask])
        return parent_var - child_var

    def predict_row(self, x, node):
        """ Parcourt l'arbre pour une seule ligne de données """
        if node.value is not None: return node.value
        if x[node.feature] <= node.threshold:
            return self.predict_row(x, node.left)
        return self.predict_row(x, node.right)

# =================================================================
# 2. LOGIQUE GLOBALE XGBOOST (Gradient Boosting)
# =================================================================
class MyXGBoost:
    """ 
    Implémentation simplifiée de l'Extreme Gradient Boosting.
    Le principe : Entraîner des arbres successifs sur les erreurs des précédents.
    """
    def __init__(self, n_estimators=10, learning_rate=0.1, max_depth=3):
        self.n_estimators = n_estimators # Nombre total d'arbres (itération)
        self.lr = learning_rate           # 'eta' : réduit l'impact de chaque arbre pour éviter l'overfitting
        self.max_depth = max_depth       # Profondeur des arbres individuels
        self.trees = []                  # Liste pour stocker tous les arbres entraînés
        self.base_pred = None             # Point de départ (Moyenne globale)

    def fit(self, X, y):
        """ Phase d'apprentissage séquentielle """
        # ÉTAPE 1 : Initialisation
        # On commence par prédire la moyenne de y pour tout le monde
        self.base_pred = np.mean(y)
        current_preds = np.full(y.shape, self.base_pred)

        print(f"Début de l'entraînement de XGBoost ({self.n_estimators} itérations)...")

        for i in range(self.n_estimators):
            # ÉTAPE 2 : Calcul des résidus (Gradients)
            # Résidu = Valeur Réelle - Prédiction Actuelle
            # L'arbre suivant va essayer de 'boucher le trou' de cette erreur.
            residuals = y - current_preds
            
            # ÉTAPE 3 : Entraînement d'un arbre sur les résidus
            tree = XGBoostTree(max_depth=self.max_depth)
            tree.fit(X, residuals)
            
            # ÉTAPE 4 : Mise à jour des prédictions globales
            # Nouvelles_Preds = Preds_Actuelles + (Learning_Rate * Prédiction_de_l_Arbre)
            for j in range(len(X)):
                current_preds[j] += self.lr * tree.predict_row(X[j], tree.root)
            
            self.trees.append(tree)
            
            # Log de progression
            mse = np.mean(np.square(y - current_preds))
            print(f"Itération {i+1}/{self.n_estimators} | Erreur Quadratique (MSE) : {mse:.4f}")

    def predict(self, X):
        """ Phase de prédiction : Somme des prédictions de tous les arbres """
        # On repart de la prédiction de base (moyenne)
        y_pred = np.full(len(X), self.base_pred)
        
        # On ajoute la contribution de chaque arbre pondérée par le learning rate
        for i in range(len(X)):
            for tree in self.trees:
                y_pred[i] += self.lr * tree.predict_row(X[i], tree.root)
        return y_pred

# =================================================================
# 3. EXÉCUTION ET VALIDATION
# =================================================================
if __name__ == "__main__":
    # Dataset : [Heures d'étude, Projets terminés] -> Note Finale
    X = np.array([[15, 2], [10, 1], [18, 3], [5, 0], [12, 1], [2, 0]])
    y = np.array([16, 12, 19, 8, 13, 5])

    # Configuration du modèle
    xgb = MyXGBoost(n_estimators=15, learning_rate=0.2, max_depth=2)
    
    # Entraînement
    xgb.fit(X, y)

    # Test sur un nouvel étudiant : 14h d'étude, 2 projets
    test = np.array([[14, 2]])
    result = xgb.predict(test)
    
    print("\n" + "="*30)
    print(f"PRÉDICTION FINALE : {result[0]:.2f}/20")
    print("="*30)

Début de l'entraînement de XGBoost (15 itérations)...
Itération 1/15 | Erreur Quadratique (MSE) : 14.2556
Itération 2/15 | Erreur Quadratique (MSE) : 9.4036
Itération 3/15 | Erreur Quadratique (MSE) : 6.2489
Itération 4/15 | Erreur Quadratique (MSE) : 4.1207
Itération 5/15 | Erreur Quadratique (MSE) : 2.7168
Itération 6/15 | Erreur Quadratique (MSE) : 1.7953
Itération 7/15 | Erreur Quadratique (MSE) : 1.1793
Itération 8/15 | Erreur Quadratique (MSE) : 0.7775
Itération 9/15 | Erreur Quadratique (MSE) : 0.5133
Itération 10/15 | Erreur Quadratique (MSE) : 0.3365
Itération 11/15 | Erreur Quadratique (MSE) : 0.2224
Itération 12/15 | Erreur Quadratique (MSE) : 0.1468
Itération 13/15 | Erreur Quadratique (MSE) : 0.0967
Itération 14/15 | Erreur Quadratique (MSE) : 0.0638
Itération 15/15 | Erreur Quadratique (MSE) : 0.0417

PRÉDICTION FINALE : 15.84/20
